In [ ]:
import pandas as pd
import numpy as np
import json
from pathlib import Path

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 200)

# Paths — adjust if your notebook runs from a different working directory
REPO_ROOT = Path.cwd().parents[1]  # week3/notebooks -> week3 -> repo root
DATA_PATH = REPO_ROOT / "data" / "processed" / "demand_enriched.parquet"

print("Repo root:", REPO_ROOT)
print("Data path:", DATA_PATH)
print("Exists:", DATA_PATH.exists())

Repo root: /Users/ananyajogalekar/Library/CloudStorage/GoogleDrive-ananyajogalekar5@gmail.com/My Drive/Duke University/03-Jobs/Campus-Jobs/AIPI-Summer-TA-Prof-Pramod-Singh/two_github_repos/Operationalizing-AI-TA
Data path: /Users/ananyajogalekar/Library/CloudStorage/GoogleDrive-ananyajogalekar5@gmail.com/My Drive/Duke University/03-Jobs/Campus-Jobs/AIPI-Summer-TA-Prof-Pramod-Singh/two_github_repos/Operationalizing-AI-TA/data/processed/demand_enriched.parquet
Exists: True


In [ ]:
df = pd.read_parquet(DATA_PATH)

print("Full data shape:", df.shape)
print("\nColumns and dtypes:")
print(df.dtypes)
print("\ntime_bucket range:", df["time_bucket"].min(), "→", df["time_bucket"].max())
print("time_bucket dtype:", df["time_bucket"].dtype)

# Split historical vs new at Jan 16, 2026
CUTOFF = pd.Timestamp("2026-01-16")

# Handle timezone if time_bucket is tz-aware
if (
    pd.api.types.is_datetime64_any_dtype(df["time_bucket"])
    and df["time_bucket"].dt.tz is not None
):
    CUTOFF = CUTOFF.tz_localize(df["time_bucket"].dt.tz)

df_historical = df[df["time_bucket"] < CUTOFF].copy()
df_new = df[df["time_bucket"] >= CUTOFF].copy()

print(f"\nHistorical shape: {df_historical.shape}")
print(
    f"  Range: {df_historical['time_bucket'].min()} → {df_historical['time_bucket'].max()}"
)
print(f"\nNew shape: {df_new.shape}")
print(f"  Range: {df_new['time_bucket'].min()} → {df_new['time_bucket'].max()}")

Full data shape: (6330245, 32)

Columns and dtypes:
PULocationID                   int64
time_bucket           datetime64[ns]
trip_count                     int64
hour                           int32
minute                         int32
dayofweek                      int32
is_weekend                      int8
month                          int32
dayofyear                      int32
weekofyear                     int64
year                           int32
slot_of_day                    int32
hour_sin                     float64
hour_cos                     float64
dow_sin                      float64
dow_cos                      float64
month_sin                    float64
month_cos                    float64
is_holiday                      int8
cbd_pricing_active              int8
borough_id                     int64
service_zone_id                int64
is_airport_zone                 int8
zone_slot_baseline           float64
lag_15min                    float32
lag_1h                 

In [3]:
print("=" * 70)
print("HISTORICAL describe()")
print("=" * 70)
print(df_historical.describe())

print("\n" + "=" * 70)
print("NEW describe()")
print("=" * 70)
print(df_new.describe())

HISTORICAL describe()
       PULocationID                    time_bucket    trip_count          hour        minute     dayofweek    is_weekend         month     dayofyear    weekofyear          year   slot_of_day  \
count  6.079392e+06                        6079392  6.079392e+06  6.079392e+06  6.079392e+06  6.079392e+06  6.079392e+06  6.079392e+06  6.079392e+06  6.079392e+06  6.079392e+06  6.079392e+06   
mean   1.456140e+02  2024-07-09 11:52:29.999988736  1.703395e+01  1.150000e+01  2.250000e+01  2.997300e+00  2.853285e-01  6.447345e+00  1.808020e+02  2.612331e+01  2.024027e+03  4.750000e+01   
min    4.000000e+00            2023-01-01 00:00:00  0.000000e+00  0.000000e+00  0.000000e+00  0.000000e+00  0.000000e+00  1.000000e+00  1.000000e+00  1.000000e+00  2.023000e+03  0.000000e+00   
25%    8.800000e+01            2023-10-05 17:56:15  2.000000e+00  5.750000e+00  1.125000e+01  1.000000e+00  0.000000e+00  3.000000e+00  8.800000e+01  1.300000e+01  2.023000e+03  2.375000e+01   
50%    1

In [ ]:
print("=" * 70)
print("NULL COUNTS — historical vs new")
print("=" * 70)
nulls = pd.DataFrame(
    {
        "historical_nulls": df_historical.isnull().sum(),
        "historical_pct": (
            df_historical.isnull().sum() / len(df_historical) * 100
        ).round(3),
        "new_nulls": df_new.isnull().sum(),
        "new_pct": (df_new.isnull().sum() / len(df_new) * 100).round(3),
    }
)
print(nulls)

NULL COUNTS — historical vs new
                    historical_nulls  historical_pct  new_nulls  new_pct
PULocationID                       0           0.000          0      0.0
time_bucket                        0           0.000          0      0.0
trip_count                         0           0.000          0      0.0
hour                               0           0.000          0      0.0
minute                             0           0.000          0      0.0
dayofweek                          0           0.000          0      0.0
is_weekend                         0           0.000          0      0.0
month                              0           0.000          0      0.0
dayofyear                          0           0.000          0      0.0
weekofyear                         0           0.000          0      0.0
year                               0           0.000          0      0.0
slot_of_day                        0           0.000          0      0.0
hour_sin           

In [ ]:
print("First 5 rows of new data:")
print(df_new.head())
print("\nLast 5 rows of new data:")
print(df_new.tail())
print("\nUnique PULocationIDs in new data:", df_new["PULocationID"].nunique())
print("Unique time buckets in new data:", df_new["time_bucket"].nunique())
print(
    "Expected (zones × buckets):",
    df_new["PULocationID"].nunique() * df_new["time_bucket"].nunique(),
)

First 5 rows of new data:
        PULocationID         time_bucket  trip_count  hour  minute  dayofweek  is_weekend  month  dayofyear  weekofyear  year  slot_of_day  hour_sin  hour_cos   dow_sin   dow_cos  month_sin  \
106656             4 2026-01-16 00:00:00           4     0       0          4           0      1         16           3  2026            0  0.000000  1.000000 -0.433884 -0.900969        0.5   
106657             4 2026-01-16 00:15:00           2     0      15          4           0      1         16           3  2026            1  0.000000  1.000000 -0.433884 -0.900969        0.5   
106658             4 2026-01-16 00:30:00           2     0      30          4           0      1         16           3  2026            2  0.000000  1.000000 -0.433884 -0.900969        0.5   
106659             4 2026-01-16 00:45:00           1     0      45          4           0      1         16           3  2026            3  0.000000  1.000000 -0.433884 -0.900969        0.5   
106660   

In [ ]:
# Issue 1: Duplicates
print("=" * 70)
print("ISSUE #1: Duplicate (PULocationID, time_bucket) rows")
print("=" * 70)

dup_mask = df.duplicated(subset=["PULocationID", "time_bucket"], keep=False)
dup_rows = df[dup_mask]

print(f"Total rows participating in duplicates: {len(dup_rows):,}")
print(f"Affected zones: {sorted(dup_rows['PULocationID'].unique())}")
print(
    f"Date range of duplicates: {dup_rows['time_bucket'].min()} → {dup_rows['time_bucket'].max()}"
)

per_zone = dup_rows.groupby("PULocationID").size().rename("dup_rows")
print("\nDuplicate rows per affected zone:")
print(per_zone)

DUP_ZONES = sorted(dup_rows["PULocationID"].unique().tolist())
DUP_START = dup_rows["time_bucket"].min()
DUP_END = dup_rows["time_bucket"].max()
DUP_COUNT = int(len(dup_rows))

ISSUE #1: Duplicate (PULocationID, time_bucket) rows
Total rows participating in duplicates: 20,170
Affected zones: [np.int64(4), np.int64(43), np.int64(87), np.int64(107), np.int64(229)]
Date range of duplicates: 2026-02-07 23:45:00 → 2026-02-28 23:45:00

Duplicate rows per affected zone:
PULocationID
4      4034
43     4034
87     4034
107    4034
229    4034
Name: dup_rows, dtype: int64


In [ ]:
# Issue 2: Out-of-range
print("=" * 70)
print("ISSUE #2: Out-of-range trip_count")
print("=" * 70)

oor_mask = (df["trip_count"] < 0) | (
    df["trip_count"] > 5000
)  # 5000 is generous upper bound
oor = df[oor_mask]

print(f"Out-of-range rows: {len(oor):,}")
print(f"min trip_count: {df['trip_count'].min()}")
print(f"max trip_count: {df['trip_count'].max()}")
print(f"\nValue distribution of bad rows:")
print(oor["trip_count"].value_counts().sort_index())
print(
    f"\nDate range of bad rows: {oor['time_bucket'].min()} → {oor['time_bucket'].max()}"
)
print(f"Affected zones (count): {oor['PULocationID'].nunique()}")

OOR_COUNT = int(len(oor))
OOR_VALUES = sorted(oor["trip_count"].unique().tolist())

ISSUE #2: Out-of-range trip_count
Out-of-range rows: 850
min trip_count: -10
max trip_count: 99999

Value distribution of bad rows:
trip_count
-10       147
-5        164
-1        186
 9999     185
 99999    168
Name: count, dtype: int64

Date range of bad rows: 2026-01-16 01:15:00 → 2026-02-28 23:15:00
Affected zones (count): 57


In [ ]:
print("=" * 70)
print("ISSUE #3: is_holiday stuck at 1")
print("=" * 70)

# Group by normalized timestamp (midnight) instead of date objects
daily = df.groupby(df["time_bucket"].dt.normalize())["is_holiday"].mean()

# Find days where rate == 1.0
stuck = daily[daily == 1.0]
print(f"Days where 100% of rows are flagged holiday: {len(stuck)}")
print(f"Range: {stuck.index.min().date()} → {stuck.index.max().date()}")

# Window around the corruption — string slicing works on DatetimeIndex
window = daily.loc["2026-01-01":"2026-02-01"]
print("\nDaily is_holiday rate, Jan 1 – Feb 1, 2026:")
print(window.to_string())

# Row count for the stuck window
stuck_dates = set(stuck.index.date)
stuck_rows = df[df["time_bucket"].dt.date.isin(stuck_dates) & (df["is_holiday"] == 1)]
print(f"\nTotal rows in the stuck window: {len(stuck_rows):,}")

HOLIDAY_STUCK_START = stuck.index.min().date()
HOLIDAY_STUCK_END = stuck.index.max().date()
HOLIDAY_STUCK_ROWS = int(len(stuck_rows))

ISSUE #3: is_holiday stuck at 1
Days where 100% of rows are flagged holiday: 50
Range: 2023-01-01 → 2026-02-16

Daily is_holiday rate, Jan 1 – Feb 1, 2026:
time_bucket
2026-01-01    1.0
2026-01-02    0.0
2026-01-03    0.0
2026-01-04    0.0
2026-01-05    0.0
2026-01-06    0.0
2026-01-07    1.0
2026-01-08    1.0
2026-01-09    1.0
2026-01-10    1.0
2026-01-11    1.0
2026-01-12    1.0
2026-01-13    1.0
2026-01-14    1.0
2026-01-15    1.0
2026-01-16    1.0
2026-01-17    1.0
2026-01-18    1.0
2026-01-19    1.0
2026-01-20    1.0
2026-01-21    1.0
2026-01-22    0.0
2026-01-23    0.0
2026-01-24    0.0
2026-01-25    0.0
2026-01-26    0.0
2026-01-27    0.0
2026-01-28    0.0
2026-01-29    0.0
2026-01-30    0.0
2026-01-31    0.0
2026-02-01    0.0

Total rows in the stuck window: 274,080


In [ ]:
# Isolate the corruption: find unusually long consecutive stretches
# Real holidays are isolated single days; corruption is a multi-day run.

stuck_dates = pd.Series(stuck.index).sort_values().reset_index(drop=True)
gaps = stuck_dates.diff().dt.days.fillna(1)
# new run starts whenever gap > 1
run_id = (gaps > 1).cumsum()

runs = (
    pd.DataFrame({"date": stuck_dates, "run": run_id})
    .groupby("run")
    .agg(start=("date", "min"), end=("date", "max"), n_days=("date", "count"))
    .sort_values("n_days", ascending=False)
)
print("Consecutive runs of fully-flagged days (longest first):")
print(runs.head(10).to_string())

# The corruption is the longest run (legitimate holidays are 1-3 day runs at most)
corrupt_run = runs.iloc[0]
print(
    f"\nCorruption window: {corrupt_run['start'].date()} → {corrupt_run['end'].date()} "
    f"({corrupt_run['n_days']} days)"
)

corrupt_mask = (
    (df["time_bucket"] >= corrupt_run["start"])
    & (df["time_bucket"] < corrupt_run["end"] + pd.Timedelta(days=1))
    & (df["is_holiday"] == 1)
)
corrupt_rows = df[corrupt_mask]
print(f"Rows in corruption window only: {len(corrupt_rows):,}")

# Update manifest variables
HOLIDAY_STUCK_START = corrupt_run["start"].date()
HOLIDAY_STUCK_END = corrupt_run["end"].date()
HOLIDAY_STUCK_ROWS = int(len(corrupt_rows))

Consecutive runs of fully-flagged days (longest first):
         start        end  n_days
run                              
34  2026-01-07 2026-01-21      15
0   2023-01-01 2023-01-01       1
26  2025-06-19 2025-06-19       1
20  2024-11-28 2024-11-28       1
21  2024-12-25 2024-12-25       1
22  2025-01-01 2025-01-01       1
23  2025-01-20 2025-01-20       1
24  2025-02-17 2025-02-17       1
25  2025-05-26 2025-05-26       1
27  2025-07-04 2025-07-04       1

Corruption window: 2026-01-07 → 2026-01-21 (15 days)
Rows in corruption window only: 82,080


In [ ]:
# Issue 4: lag_1week contamination
print("=" * 70)
print("ISSUE #4: lag_1week feature broken for some zones")
print("=" * 70)

# Use clean historical period only (lags should be reliable here, no corruption in lag itself)
hist = df[(df["time_bucket"] < CUTOFF) & df["lag_1week"].notna()]

corrs = (
    hist.groupby("PULocationID")
    .apply(lambda g: g["lag_1week"].corr(g["trip_count"]))
    .rename("corr_lag1w_target")
    .sort_values()
)

print("Bottom 10 zones by lag_1week ↔ trip_count correlation:")
print(corrs.head(10))
print("\nTop 5 zones (for reference):")
print(corrs.tail(5))

# Threshold: healthy zones correlate ~0.85; broken zones drop to near 0
broken = corrs[corrs < 0.20]
print(f"\nZones with broken lag_1week correlation (< 0.20): {broken.index.tolist()}")
print(f"Median correlation across all zones: {corrs.median():.3f}")

LAG_BROKEN_ZONES = broken.index.tolist()
LAG_THRESHOLD = 0.20

ISSUE #4: lag_1week feature broken for some zones
Bottom 10 zones by lag_1week ↔ trip_count correlation:
PULocationID
93     0.327289
74     0.360070
45     0.392766
195    0.409064
41     0.413180
24     0.489683
209    0.502407
70     0.516734
50     0.530200
88     0.565459
Name: corr_lag1w_target, dtype: float64

Top 5 zones (for reference):
PULocationID
79     0.866678
140    0.871218
148    0.878911
237    0.892036
236    0.892177
Name: corr_lag1w_target, dtype: float64

Zones with broken lag_1week correlation (< 0.20): []
Median correlation across all zones: 0.772


/var/folders/yw/zr51wswj5pl5h03k4z3_qznr0000gn/T/ipykernel_6270/2559097434.py:11: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda g: g['lag_1week'].corr(g['trip_count']))


In [ ]:
# write manifest
manifest = {
    "baseline_period": {"start": "2023-01-01", "end": "2026-01-15"},
    "new_period": {"start": "2026-01-16", "end": str(df["time_bucket"].max().date())},
    "issues": {
        "duplicates": {
            "affected_zones": DUP_ZONES,
            "period_start": str(DUP_START),
            "period_end": str(DUP_END),
            "row_count": DUP_COUNT,
            "natural_key": ["PULocationID", "time_bucket"],
        },
        "out_of_range_trip_count": {
            "row_count": OOR_COUNT,
            "bad_values": OOR_VALUES,
            "valid_range": [0, 5000],
        },
        "is_holiday_stuck": {
            "period_start": str(HOLIDAY_STUCK_START),
            "period_end": str(HOLIDAY_STUCK_END),
            "row_count": HOLIDAY_STUCK_ROWS,
            "expected_rate_max": 0.10,
        },
        "lag_1week_cross_contamination": {
            "affected_zones": LAG_BROKEN_ZONES,
            "correlation_threshold": LAG_THRESHOLD,
            "expected_correlation_min": 0.50,
        },
    },
}

manifest_path = REPO_ROOT / "data" / "processed" / "week3_corruption_manifest.json"
with open(manifest_path, "w") as f:
    json.dump(manifest, f, indent=2, default=str)

print(f"Wrote manifest → {manifest_path}")
print(json.dumps(manifest, indent=2, default=str))